In [2]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [3]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

rawDF = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .csv("hdfs://localhost:9000/user/talentum/silver_csv")

In [ ]:
silverDF = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("file:///home/talentum/Downloads/silver_csv_single")

In [ ]:
from pyspark.sql.functions import regexp_replace

typedDF = rawDF \
.withColumn("Applicants_Count",
            regexp_replace(col("Applicants_Count"), "[^0-9]", "").cast(IntegerType())) \
.withColumn("Interview_Rounds",
            regexp_replace(col("Interview_Rounds"), "[^0-9]", "").cast(IntegerType())) \
.withColumn("Number_of_Openings",
            regexp_replace(col("Number_of_Openings"), "[^0-9]", "").cast(IntegerType())) \
.withColumn("Salary_Min",
            regexp_replace(col("Salary_Min"), "[^0-9]", "").cast(IntegerType())) \
.withColumn("Salary_Max",
            regexp_replace(col("Salary_Max"), "[^0-9]", "").cast(IntegerType())) \
.withColumn("Selection_Rate",
            regexp_replace(col("Selection_Rate"), "[^0-9.]", "").cast(DoubleType())) \
.withColumn("Posted_Date",
            to_date(col("Posted_Date"), "M/d/yyyy")) \
.withColumn("Application_Deadline",
            to_date(col("Application_Deadline"), "M/d/yyyy"))

In [ ]:
typedDF.printSchema()

In [ ]:
typedDF.write \
.mode("overwrite") \
.parquet("hdfs://localhost:9000/user/talentum/silver_typed")

In [ ]:
silverDF = spark.read.parquet(
    "hdfs://localhost:9000/user/talentum/silver_typed"
)

In [ ]:
silverDF.printSchema()

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import DateType

spark = SparkSession.builder \
    .appName("Silver Cleaning") \
    .getOrCreate()

# ======================================================
# Read Typed Parquet
# ======================================================

silverDF = spark.read.parquet(
    "hdfs://localhost:9000/user/talentum/silver_typed"
)

print("Rows Before Cleaning :", silverDF.count())

# ======================================================
# Remove Duplicate Records
# ======================================================

silverDF = silverDF.dropDuplicates()

# ======================================================
# Fill String Columns with Mode
# ======================================================

string_cols = [
    "Benefits",
    "Company_Name",
    "Currency",
    "Education_Level",
    "Employment_Type",
    "Experience_Level",
    "Industry",
    "Job_Category",
    "Job_Description",
    "Job_Status",
    "Job_Title",
    "Location_City",
    "Location_Country",
    "Location_State",
    "Preferred_Skills",
    "Recruiter_Email",
    "Recruiter_Name",
    "Remote_Option",
    "Required_Skills",
    "Source",
    "Work_Shift"
]

for c in string_cols:

    mode_row = (
        silverDF
        .filter(col(c).isNotNull())
        .groupBy(c)
        .count()
        .orderBy(desc("count"))
        .first()
    )

    if mode_row:
        silverDF = silverDF.fillna({c: mode_row[0]})

# ======================================================
# Fill Numeric Columns with Median
# ======================================================

numeric_cols = [
    "Applicants_Count",
    "Interview_Rounds",
    "Number_of_Openings",
    "Salary_Min",
    "Salary_Max",
    "Selection_Rate"
]

for c in numeric_cols:

    median = silverDF.approxQuantile(c, [0.5], 0.01)[0]

    silverDF = silverDF.fillna({c: median})

# ======================================================
# Fill Date Columns with Most Frequent Date
# ======================================================

posted_date = (
    silverDF
    .filter(col("Posted_Date").isNotNull())
    .groupBy("Posted_Date")
    .count()
    .orderBy(desc("count"))
    .first()
)

if posted_date:
    silverDF = silverDF.withColumn(
        "Posted_Date",
        when(
            col("Posted_Date").isNull(),
            lit(posted_date[0]).cast(DateType())
        ).otherwise(col("Posted_Date"))
    )

deadline_date = (
    silverDF
    .filter(col("Application_Deadline").isNotNull())
    .groupBy("Application_Deadline")
    .count()
    .orderBy(desc("count"))
    .first()
)

if deadline_date:
    silverDF = silverDF.withColumn(
        "Application_Deadline",
        when(
            col("Application_Deadline").isNull(),
            lit(deadline_date[0]).cast(DateType())
        ).otherwise(col("Application_Deadline"))
    )

# ======================================================
# Verify NULL Values
# ======================================================

print("\nNULL Values After Cleaning\n")

silverDF.select([
    sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in silverDF.columns
]).show(vertical=True)

# ======================================================
# Row Count
# ======================================================

print("Rows After Cleaning :", silverDF.count())

# ======================================================
# Preview
# ======================================================

silverDF.show(10, False)

# ======================================================
# Save Silver Layer
# ======================================================

silverDF.write \
    .mode("overwrite") \
    .parquet("hdfs://localhost:9000/user/talentum/silver_cleaned")

print("========================================")
print("Silver Cleaning Completed Successfully")
print("Output Path : /user/talentum/silver_cleaned")
print("========================================")

spark.stop()

In [ ]:
from pyspark.sql.functions import *

# Read Silver Cleaned Data
silverDF = spark.read.parquet(
    "hdfs://localhost:9000/user/talentum/silver_cleaned"
)

# ------------------------------------------------------
# Total Rows
# ------------------------------------------------------
print("Total Rows :", silverDF.count())

# ------------------------------------------------------
# Total Columns
# ------------------------------------------------------
print("Total Columns :", len(silverDF.columns))

# ------------------------------------------------------
# Check NULL values in each column
# ------------------------------------------------------
null_df = silverDF.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in silverDF.columns
])

null_df.show(vertical=True)

# ------------------------------------------------------
# Count columns that still contain NULL values
# ------------------------------------------------------
null_counts = null_df.collect()[0].asDict()

bad_columns = {k: v for k, v in null_counts.items() if v > 0}

print("="*60)
print("Columns Still Having NULL Values")
print("="*60)

if bad_columns:
    for k, v in bad_columns.items():
        print(f"{k} : {v}")
else:
    print("No NULL values found.")

print("="*60)
print("Total Bad Columns :", len(bad_columns))
print("="*60)

In [ ]:
silverDF.printSchema()

print("Duplicate Rows :", silverDF.count() - silverDF.dropDuplicates().count())

silverDF.describe().show()

In [ ]:
silverDF.describe(
    "Applicants_Count",
    "Salary_Min",
    "Salary_Max",
    "Selection_Rate",
    "Interview_Rounds",
    "Number_of_Openings"
).show()

In [ ]:
silverDF = spark.read.parquet(
    "hdfs://localhost:9000/user/talentum/silver_cleaned"
)

In [5]:
from pyspark.sql.functions import *
from pyspark.sql.types import DateType

# =====================================================
# 1. Remove Duplicate Records
# =====================================================
silverDF = silverDF.dropDuplicates()

# =====================================================
# 2. Replace Invalid String Values with NULL
# =====================================================

invalid_values = [
    "!!INVALID!!",
    "void",
    "unknown",
    "UNKNOWN",
    "null",
    "NULL",
    "N/A",
    "NA",
    ""
]

string_cols = [
    "Benefits","Company_Name","Currency","Education_Level",
    "Employment_Type","Experience_Level","Industry",
    "Job_Category","Job_Description","Job_Status",
    "Job_Title","Location_City","Location_Country",
    "Location_State","Preferred_Skills",
    "Recruiter_Email","Recruiter_Name",
    "Remote_Option","Required_Skills",
    "Source","Work_Shift"
]

for c in string_cols:
    silverDF = silverDF.withColumn(
        c,
        when(trim(col(c)).isin(invalid_values), None)
        .otherwise(col(c))
    )

# =====================================================
# 3. Replace Invalid Numeric Values with NULL
# =====================================================

silverDF = silverDF.withColumn(
    "Applicants_Count",
    when((col("Applicants_Count")==2147483647) | (col("Applicants_Count")<0), None)
    .otherwise(col("Applicants_Count"))
)

silverDF = silverDF.withColumn(
    "Interview_Rounds",
    when(
        (col("Interview_Rounds")==2147483647) |
        (col("Interview_Rounds")>20) |
        (col("Interview_Rounds")<0),
        None
    ).otherwise(col("Interview_Rounds"))
)

silverDF = silverDF.withColumn(
    "Number_of_Openings",
    when(
        (col("Number_of_Openings")==2147483647) |
        (col("Number_of_Openings")>10000) |
        (col("Number_of_Openings")<0),
        None
    ).otherwise(col("Number_of_Openings"))
)

silverDF = silverDF.withColumn(
    "Salary_Min",
    when(
        (col("Salary_Min")==2147483647) |
        (col("Salary_Min")<0),
        None
    ).otherwise(col("Salary_Min"))
)

silverDF = silverDF.withColumn(
    "Salary_Max",
    when(
        (col("Salary_Max")==2147483647) |
        (col("Salary_Max")<0),
        None
    ).otherwise(col("Salary_Max"))
)

silverDF = silverDF.withColumn(
    "Selection_Rate",
    when(
        (col("Selection_Rate")>100) |
        (col("Selection_Rate")<0),
        None
    ).otherwise(col("Selection_Rate"))
)

# =====================================================
# 4. Fill String NULLs with Mode
# =====================================================

for c in string_cols:

    row = (
        silverDF
        .filter(col(c).isNotNull())
        .groupBy(c)
        .count()
        .orderBy(desc("count"))
        .first()
    )

    if row:
        silverDF = silverDF.fillna({c: row[0]})

# =====================================================
# 5. Fill Numeric NULLs with Median
# =====================================================

numeric_cols = [
    "Applicants_Count",
    "Interview_Rounds",
    "Number_of_Openings",
    "Salary_Min",
    "Salary_Max",
    "Selection_Rate"
]

for c in numeric_cols:

    median = silverDF.approxQuantile(c,[0.5],0.01)[0]

    if median is not None:
        silverDF = silverDF.fillna({c: median})

# =====================================================
# 6. Fill Date NULLs with Most Frequent Date
# =====================================================

posted = (
    silverDF
    .filter(col("Posted_Date").isNotNull())
    .groupBy("Posted_Date")
    .count()
    .orderBy(desc("count"))
    .first()
)

deadline = (
    silverDF
    .filter(col("Application_Deadline").isNotNull())
    .groupBy("Application_Deadline")
    .count()
    .orderBy(desc("count"))
    .first()
)

if posted:
    silverDF = silverDF.withColumn(
        "Posted_Date",
        coalesce(
            col("Posted_Date"),
            lit(posted[0]).cast(DateType())
        )
    )

if deadline:
    silverDF = silverDF.withColumn(
        "Application_Deadline",
        coalesce(
            col("Application_Deadline"),
            lit(deadline[0]).cast(DateType())
        )
    )

# =====================================================
# 7. Verify NULL Values
# =====================================================

print("Remaining NULL Values")

silverDF.select([
    sum(when(col(c).isNull(),1).otherwise(0)).alias(c)
    for c in silverDF.columns
]).show(vertical=True)

# =====================================================
# 8. Verify Duplicate Rows
# =====================================================

print(
    "Duplicate Rows:",
    silverDF.count()-silverDF.dropDuplicates().count()
)

# =====================================================
# 9. Dataset Statistics
# =====================================================

silverDF.describe().show()

# =====================================================
# 10. Save Cleaned Silver Layer
# =====================================================

silverDF.write \
.mode("overwrite") \
.parquet("hdfs://localhost:9000/user/talentum/silver_cleaned")

print("Silver Cleaning Completed Successfully!")

Remaining NULL Values
-RECORD 0-------------------
 Applicants_Count     | 0   
 Application_Deadline | 0   
 Benefits             | 0   
 Company_Name         | 0   
 Currency             | 0   
 Education_Level      | 0   
 Employment_Type      | 0   
 Experience_Level     | 0   
 Industry             | 0   
 Interview_Rounds     | 0   
 Job_Category         | 0   
 Job_Description      | 0   
 Job_ID               | 0   
 Job_Status           | 0   
 Job_Title            | 0   
 Location_City        | 0   
 Location_Country     | 0   
 Location_State       | 0   
 Number_of_Openings   | 0   
 Posted_Date          | 0   
 Preferred_Skills     | 0   
 Recruiter_Email      | 0   
 Recruiter_Name       | 0   
 Remote_Option        | 0   
 Required_Skills      | 0   
 Salary_Max           | 0   
 Salary_Min           | 0   
 Selection_Rate       | 0   
 Source               | 0   
 Work_Shift           | 0   

Duplicate Rows: 0
+-------+--------------------+-----------------+------------

In [8]:
from pyspark.sql.functions import col

silverDF = silverDF.filter(
    (col("Applicants_Count") > 0) &
    (col("Applicants_Count") <= 100000) &

    (col("Salary_Min") > 0) &
    (col("Salary_Max") > 0) &
    (col("Salary_Max") >= col("Salary_Min")) &
    (col("Salary_Min") <= 5000000) &
    (col("Salary_Max") <= 5000000)
)

print("Silver Cleaning Completed Successfully!")

silverDF.select(
    "Applicants_Count",
    "Salary_Min",
    "Salary_Max"
).describe().show()

Silver Cleaning Completed Successfully!
+-------+-----------------+------------------+-----------------+
|summary| Applicants_Count|        Salary_Min|       Salary_Max|
+-------+-----------------+------------------+-----------------+
|  count|           428653|            428653|           428653|
|   mean|898.5841858099675|287821.39999020187| 420809.914539266|
| stddev|4989.065248318087| 432280.2507274276|592658.8666371749|
|    min|                1|                 1|                1|
|    max|            99999|           4949055|          4999000|
+-------+-----------------+------------------+-----------------+



In [9]:
silverDF.write.mode("overwrite").parquet("/home/talentum/silver/job_market")